In [ ]:
import torch
import os
from pathlib import Path

# Get the notebook's directory and go up one level to project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Goes up from /notebooks to root
DATASET_FOLDER = "license plate dataset"
DATASET_PATH = PROJECT_ROOT / DATASET_FOLDER

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if DATASET_PATH.exists():
    print(f"\nContents: {os.listdir(DATASET_PATH)}")
    
    # Check train folder
    train_path = DATASET_PATH / "train" / "images"
    print(f"\nTrain images exist: {train_path.exists()}")
    if train_path.exists():
        print(f"Sample files: {os.listdir(train_path)[:3]}")
else:
    print("❌ Dataset not found! Check folder name.")
    
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU (expect slow training)'}")


Project root: d:\unishit\terme 10\Data Mining\License Plate Recognition Project\License Plate Recognition
Dataset path: d:\unishit\terme 10\Data Mining\License Plate Recognition Project\License Plate Recognition\license plate dataset
Dataset exists: True

Contents: ['test', 'train', 'valid']

Train images exist: True
Sample files: ['series1_06a83a27-010_jpg.rf.1c7c31738a2abc484de81bddb2a6f591.jpg', 'series1_0ccda24d-164_JPG.rf.3bc7ab570f0296d9dbc3b24b861a97df.jpg', 'series1_0cea2ea6-154_JPG.rf.a911688724ddf95d17766701bfdd1ac0.jpg']

PyTorch: 2.9.1+cpu
CUDA available: False
Device: CPU (expect slow training)


In [ ]:
import yaml

# Build absolute path with forward slashes (YOLO prefers this even on Windows)
dataset_abs_path = str(DATASET_PATH.resolve()).replace("\\", "/")

config = {
    'path': dataset_abs_path,  # Absolute path to dataset
    'train': 'train/images',
    'val': 'valid/images',     # Note: your folder is named 'valid' not 'val'
    'test': 'test/images',
    'names': {
        0: 'plate'
    }
}

# Save yaml in project root (next to dataset folder)
yaml_path = PROJECT_ROOT / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Created: {yaml_path}")
print(f"\nContent:")
print(open(yaml_path, 'r').read())



✅ Created: d:\unishit\terme 10\Data Mining\License Plate Recognition Project\License Plate Recognition\data.yaml

Content:
names:
  0: plate
path: D:/unishit/terme 10/Data Mining/License Plate Recognition Project/License Plate
  Recognition/license plate dataset
test: test/images
train: train/images
val: valid/images



In [4]:
from ultralytics import YOLO
import time

print("⏳ Loading model...")
# Use yolo11n (nano) - smallest and fastest for CPU
model = YOLO('yolo11n.pt')

print("🚀 Starting training...")
print("⚠️  WARNING: This will be SLOW on CPU (~1-2 min per epoch)")
print("💡 Tip: You can interrupt (Ctrl+C) anytime - weights save every epoch\n")

start_time = time.time()

# Train with CPU-optimized settings
results = model.train(
    data=str(PROJECT_ROOT / 'data.yaml'),  # Path to your YAML
    epochs=20,              # Start with 20 for testing, change to 100 later
    imgsz=640,
    batch=4,                # MAX for CPU (reduce to 2 if RAM error)
    device='cpu',           # Force CPU
    workers=0,              # REQUIRED for Windows (prevents freezing)
    name='ir_plate_cpu',    # Run name
    exist_ok=True,          # Overwrite previous run with same name
    patience=10,            # Stop if no improvement for 10 epochs
    verbose=True
)

elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes!")
print(f"📊 Best mAP@0.5: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

# Show where weights are saved
weights_path = PROJECT_ROOT / 'runs' / 'detect' / 'ir_plate_cpu' / 'weights' / 'best.pt'
print(f"💾 Best model saved to: {weights_path}")
print(f"   Exists: {weights_path.exists()}")


⏳ Loading model...
🚀 Starting training...
⚠️  WARNING: This will be SLOW on CPU (~1-2 min per epoch)
💡 Tip: You can interrupt (Ctrl+C) anytime - weights save every epoch

New https://pypi.org/project/ultralytics/8.4.63 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.62  Python-3.12.3 torch-2.9.1+cpu CPU (AMD Ryzen 5 PRO 4650U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\unishit\terme 10\Data Mining\License Plate Recognition Project\License Plate Recognition\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hal

In [5]:
import shutil
from pathlib import Path

# Source (where it actually is)
src = Path("runs/detect/ir_plate_cpu/weights/best.pt")
# Destination (project root for easy access)
dst = Path("../../best_plate_detector.pt")

if src.exists():
    shutil.copy(src, dst)
    print(f"✅ Model copied to: {dst.resolve()}")
else:
    print("❌ Not found in notebooks/runs/, checking project root...")
    # Try alternative location
    src = Path("../../runs/detect/ir_plate_cpu/weights/best.pt")
    if src.exists():
        shutil.copy(src, dst)
        print(f"✅ Model copied to: {dst.resolve()}")


✅ Model copied to: D:\unishit\terme 10\Data Mining\License Plate Recognition Project\best_plate_detector.pt
